In [29]:
# use only if in colab
from google.colab import drive
import os
drive.mount('/content/drive')
pth = '/content/drive/MyDrive/Jane Austen LSTM'
os.chdir(pth)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
# load the libraries and define some helper function
# no need to dig into this part

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import numpy as np
import tensorflow as tf

def generate_text(model, start_string, char_to_idx, idx_to_char, generation_length=1000, temperature=1.0):

    # Converting our start string to numbers (vectorizing)
    input_eval = [char_to_idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    # Empty string to store our results
    text_generated = []

    # Low temperatures results in more predictable text.
    # Higher temperatures results in more surprising text.
    # Experiment to find the best setting.
    model.reset_states()
    for i in range(generation_length):
        predictions = model(input_eval)
        # remove the batch dimension
        predictions = tf.squeeze(predictions, 0)

        # using a categorical distribution to predict the character returned by the model
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()

        # We pass the predicted character as the next input to the model
        # along with the previous hidden state
        input_eval = tf.expand_dims([predicted_id], 0)
        text_generated.append(idx_to_char[predicted_id])

    return (start_string + ''.join(text_generated))

def build_base_vocabulary():
    file_path = 'pride_and_prejudice.txt'
    text = open(file_path, 'rb').read().decode(encoding='utf-8')

    # Create a mapping from unique characters to indices
    vocab = sorted(set(text))
    char_to_idx = {char: idx for idx, char in enumerate(vocab)}
    idx_to_char = np.array(vocab)
    return char_to_idx,idx_to_char

def build_full_vocabulary():
    file_path = 'works_of_jane_austen.txt'
    text = open(file_path, 'rb').read().decode(encoding='utf-8')

    # Create a mapping from unique characters to indices
    vocab = sorted(set(text))
    char_to_idx = {char: idx for idx, char in enumerate(vocab)}
    idx_to_char = np.array(vocab)
    return char_to_idx,idx_to_char

def fancy_print(flag, result_1, result_2):
    if flag == 'base':
        heading = "Text generated by model trained on one novel: Pride and Prejudice"
    elif flag == 'full':
        heading = "Text generated by model trained on all of Jane Austen's novels"
    elif flag == 'gpt4':
        heading = "Text generated by GPT-4 pretending to be Jane Austen"
    else: return 0
    print(heading)
    print("\n"+"-"*10+"Result 1"+"-"*10)
    print(result_1)
    print("\n"+"-"*10+"Result 2"+"-"*10)
    print(result_2)

# Text Generation with LSTMs

## Write like Jane Austen

In this notebook, we will aim to train a neural network that can write like Jane Austen. Isn't it cool!? In order to do so, we'll first train an LSTM using only Pride and Prejudice and then adding more of Jane Austen's work to the training data to see if our model improves in performance.

## Model Performance Comparison

In this section, we load pre-trained weights for our LSTM models and generate text to compare the performance differences attributed to varying training dataset sizes. Below are the results from models trained with different extents of data:

- `model_base`: Trained on "Pride and Prejudice" only .
- `model_full`: Trained on the complete works of Jane Austen.

### Model trained on one novel: Pride and Prejudice

In [61]:
base_model = tf.keras.models.load_model('janeausten_base.keras')
char_to_idx,idx_to_char = build_base_vocabulary()

In [68]:
seed_1 = "It is a truth universally acknowledged, "
seed_2 = "I may have lost my heart, "
base_result_1 = generate_text(base_model, seed_1, char_to_idx, idx_to_char, generation_length=200, temperature=0.5)
base_result_2 = generate_text(base_model, seed_2, char_to_idx, idx_to_char, generation_length=200, temperature=0.5)

In [69]:
fancy_print('base',base_result_1,base_result_2)

Text generated by model trained on one novel: Pride and Prejudice

----------Result 1----------
It is a truth universally acknowledged, that had given them so many
previous months of suspense and vexation.

"And this is a great deal of him."

"It is a fail mile from Miss Darcy, and was dismissed from her charge of her worth, and when 

----------Result 2----------
I may have lost my heart, as I do not
reflect when the very sight of either, that she could have less to see them; but still they were to have an
occupied them. But I can assure you there is quite a settled
thing; and as soon 


### Model trained on all of Jane Austen's novels.

In [70]:
full_model = tf.keras.models.load_model('janeausten_full.keras')
char_to_idx,idx_to_char = build_full_vocabulary()

In [71]:
full_result_1 = generate_text(full_model, seed_1, char_to_idx, idx_to_char, generation_length=200, temperature=0.5)
full_result_2 = generate_text(full_model, seed_2, char_to_idx, idx_to_char, generation_length=200, temperature=0.5)
fancy_print('full',full_result_1,full_result_2)

Text generated by model trained on all of Jane Austen's novels

----------Result 1----------
It is a truth universally acknowledged, and her daughter and her father,
though whether it was a favour of the party, to an end of the man were staying to the same
that to her again, and the inclination to his sister's intimacy, she was r

----------Result 2----------
I may have lost my heart, and the last act of her brother,
and has been so resolution of her uncle was a recollection of a complexion,
and her heart accounts and sister were obliged to make her desire before the
ended the p


### GPT 4.0
We'll give ChatGPT the following prompt along with our previous seed texts:

Write like Jane Austen for 200 characters. Start with the following text: {seed_text}

In [81]:
pip install openai

In [73]:
os.environ["OPENAI_API_KEY"] = "your api key here"

In [79]:
import os
from openai import OpenAI

def write_like_jane_austen(seed_text):
    client = OpenAI(
        # This is the default and can be omitted
        api_key=os.environ.get("OPENAI_API_KEY"),
    )

    response = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": f"Write like Jane Austen for 200 characters. Start with the following text: {seed_text}",
            }
        ],
        model="gpt-4",
        temperature=0,
        top_p=0.95,
        max_tokens=1024
    )
    return response.choices[0].message.content


In [80]:
GPT_result_1 = write_like_jane_austen(seed_1)
GPT_result_2 = write_like_jane_austen(seed_2)
fancy_print('gpt4',GPT_result_1,GPT_result_2)

Text generated by GPT-4 pretending to be Jane Austen

----------Result 1----------
that a single man in possession of a good fortune, must be in want of a wife. Yet, such a man, when introduced into society, often finds himself the object of many a speculative gaze.

----------Result 2----------
I may have lost my heart, but I have not lost my senses. I am not so easily swayed by the whims of passion, nor so readily beguiled by the allure of a handsome countenance.


# Complete Code
Now let's delve deeper into how we built the model!

## Load and Prepare the Data
We'll first load the text data, then create a mapping of characters to integers.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint
import os

# Load text data
file_path = 'pride_and_prejudice.txt'
text = open(file_path, 'rb').read().decode(encoding='utf-8')

# Create a mapping from unique characters to indices
vocab = sorted(set(text))
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = np.array(vocab)

# Encode text as integer array
text_as_int = np.array([char_to_idx[char] for char in text])


## Create Training Examples and Targets
We will split the text into sequences of a fixed length and use these sequences as inputs to the model.

In [ ]:
# Set the maximum length sentence we want for a single input
seq_length = 100
examples_per_epoch = len(text) // (seq_length + 1)

# Create training examples / targets
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)


## Create Training Batches

In [ ]:
# Batch size
BATCH_SIZE = 64

# Buffer size to shuffle the dataset
BUFFER_SIZE = 10000

dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)


## Build the Model
We use an LSTM model with an Embedding layer to process the text data.

In [ ]:
# Length of the vocabulary in chars
vocab_size = len(vocab)

# Embedding dimension
embedding_dim = 256

# Number of RNN units
rnn_units = 1024

def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = Sequential([
        Embedding(vocab_size, embedding_dim, batch_input_shape=[batch_size, None]),
        LSTM(rnn_units, return_sequences=True, stateful=True, recurrent_initializer='glorot_uniform'),
        Dense(vocab_size)
    ])
    return model

model = build_model(vocab_size=len(vocab), embedding_dim=embedding_dim, rnn_units=rnn_units, batch_size=BATCH_SIZE)


In [ ]:
def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

model.compile(optimizer='adam', loss=loss)


## Best Practices: Checkpoint
Using checkpoints and saving the model is generally considered a very good practice in the field of machine learning and deep learning for several important reasons:

* Preventing Data Loss: Training models, especially deep learning models like LSTMs, can be time-consuming and resource-intensive. Checkpoints prevent loss of progress in case of interruptions like power failures or system crashes.

* Model Evaluation and Comparison: Saving models at different stages of training (or with different architectures) allows you to compare their performance on the validation set. This helps in selecting the best model for your task.

* Early Stopping: Checkpoints can be used in conjunction with early stopping, where training is halted as soon as the model performance begins to degrade on a validation set. This helps prevent overfitting.

* Continuing Training: If you decide to train your model further, checkpoints allow you to resume training from a specific point rather than starting over.

* Experimentation: Having saved models allows you to experiment with different aspects of your model (like hyperparameters) without losing your previous work.

In [ ]:
# Directory where the checkpoints will be saved
checkpoint_dir = 'checkpoint_base'
# Name of the checkpoint files
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}")

checkpoint_callback = ModelCheckpoint(filepath=checkpoint_prefix, save_weights_only=True)


## Train the model
You can try to train this model in colab with a GPU runtime and it should be light weight enough to train in 30 minutes.

In [ ]:
EPOCHS = 100
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])


Epoch 1/100


I0000 00:00:1713311222.200032   18345 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


105/105 [==============================] - 4s 19ms/step - loss: 2.9203
Epoch 2/100
105/105 [==============================] - 2s 15ms/step - loss: 2.3093
Epoch 3/100
105/105 [==============================] - 2s 15ms/step - loss: 2.0456
Epoch 4/100
105/105 [==============================] - 2s 15ms/step - loss: 1.8373
Epoch 5/100
105/105 [==============================] - 2s 15ms/step - loss: 1.6844
Epoch 6/100
105/105 [==============================] - 2s 15ms/step - loss: 1.5685
Epoch 7/100
105/105 [==============================] - 2s 15ms/step - loss: 1.4792
Epoch 8/100
105/105 [==============================] - 2s 15ms/step - loss: 1.4117
Epoch 9/100
105/105 [==============================] - 2s 15ms/step - loss: 1.3583
Epoch 10/100
105/105 [==============================] - 2s 15ms/step - loss: 1.3161
Epoch 11/100
105/105 [==============================] - 2s 15ms/step - loss: 1.2811
Epoch 12/100
105/105 [==============================] - 2s 15ms/step - loss: 1.2525
Epoch 13/100


## Rebuild the Model
First, we need to rebuild the model with a batch size of 1 so it can process one piece of text at a time. This is necessary because the model was originally trained with a batch size greater than 1.

In [ ]:
model = build_model(vocab_size=len(vocab), embedding_dim=256, rnn_units=1024, batch_size=1)

# Load the weights from the last checkpoint after training
model.load_weights(tf.train.latest_checkpoint(checkpoint_dir))
model.build(tf.TensorShape([1, None]))


## Text Generation Function
This function will generate text based on a seed text input. You can specify the length of the generated text and modify the temperature parameter to tweak randomness.

In [ ]:
def generate_text(model, start_string, generation_length=1000, temperature=1.0):
    # Converting our start string to numbers (vectorizing)
    input_eval = [char_to_idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    # Empty string to store our results
    text_generated = []

    # Low temperatures results in more predictable text.
    # Higher temperatures results in more surprising text.
    # Experiment to find the best setting.
    model.reset_states()
    for i in range(generation_length):
        predictions = model(input_eval)
        # remove the batch dimension
        predictions = tf.squeeze(predictions, 0)

        # using a categorical distribution to predict the character returned by the model
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()

        # We pass the predicted character as the next input to the model
        # along with the previous hidden state
        input_eval = tf.expand_dims([predicted_id], 0)
        text_generated.append(idx_to_char[predicted_id])

    return (start_string + ''.join(text_generated))


## Try it out

In [ ]:
seed_text = "It is a truth universally acknowledged, "
print(generate_text(model, start_string=seed_text, generation_length=200))


It is a truth universally acknowledged, had been to secure an
importance. Nor amusement; in expressing it, but her
own nephew, in her air
and manner not oppose such an injunction
that would robsent was settled by a ride on the but began to all
that he is to lose what he er they will settle it must be
amused against them, and at leisure to observe the reason, she decled of it. I never saw it ought
on that account, or that Marcas of course. He was quite getts to her,
and at consider it all, and resolved to avoid a
conversation with ever


## Save the model for future use

In [ ]:
model.save("janeausten_base.keras")

## Same code but for used to train model on the complete works of Jane Austen
Below are essentially the same code as above but used to train model on the complete works of Jane Austen

In [ ]:
# Load text data
file_path = 'works_of_jane_austen.txt'
text = open(file_path, 'rb').read().decode(encoding='utf-8')

# Create a mapping from unique characters to indices
vocab = sorted(set(text))
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = np.array(vocab)

# Encode text as integer array
text_as_int = np.array([char_to_idx[char] for char in text])

# Set the maximum length sentence we want for a single input
seq_length = 100
examples_per_epoch = len(text) // (seq_length + 1)

# Create training examples / targets
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

# Batch size
BATCH_SIZE = 64

# Buffer size to shuffle the dataset
BUFFER_SIZE = 10000

dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

# Length of the vocabulary in chars
vocab_size = len(vocab)

# Embedding dimension
embedding_dim = 256

# Number of RNN units
rnn_units = 1024

def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = Sequential([
        Embedding(vocab_size, embedding_dim, batch_input_shape=[batch_size, None]),
        LSTM(rnn_units, return_sequences=True, stateful=True, recurrent_initializer='glorot_uniform'),
        Dense(vocab_size)
    ])
    return model

model = build_model(vocab_size=len(vocab), embedding_dim=embedding_dim, rnn_units=rnn_units, batch_size=BATCH_SIZE)

def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

model.compile(optimizer='adam', loss=loss)

# Directory where the checkpoints will be saved
checkpoint_dir = 'checkpoint_full'
# Name of the checkpoint files
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}")

checkpoint_callback = ModelCheckpoint(filepath=checkpoint_prefix, save_weights_only=True)

EPOCHS = 100
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])

Epoch 1/100


I0000 00:00:1713312332.672428   28017 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


685/685 [==============================] - 12s 14ms/step - loss: 1.8225
Epoch 2/100
685/685 [==============================] - 10s 13ms/step - loss: 1.2581
Epoch 3/100
685/685 [==============================] - 10s 13ms/step - loss: 1.1736
Epoch 4/100
685/685 [==============================] - 10s 13ms/step - loss: 1.1323
Epoch 5/100
685/685 [==============================] - 10s 14ms/step - loss: 1.1040
Epoch 6/100
685/685 [==============================] - 10s 13ms/step - loss: 1.0819
Epoch 7/100
685/685 [==============================] - 10s 13ms/step - loss: 1.0622
Epoch 8/100
685/685 [==============================] - 10s 13ms/step - loss: 1.0449
Epoch 9/100
685/685 [==============================] - 10s 13ms/step - loss: 1.0290
Epoch 10/100
685/685 [==============================] - 10s 13ms/step - loss: 1.0140
Epoch 11/100
685/685 [==============================] - 18s 25ms/step - loss: 0.9992
Epoch 12/100
685/685 [==============================] - 19s 27ms/step - loss: 0.9859
E

In [ ]:
model = build_model(vocab_size=len(vocab), embedding_dim=256, rnn_units=1024, batch_size=1)

# Load the weights from the last checkpoint after training
model.load_weights(tf.train.latest_checkpoint(checkpoint_dir))
model.build(tf.TensorShape([1, None]))

In [ ]:
seed_text = "It is a truth universally acknowledged, "
print(generate_text(model, start_string=seed_text, generation_length=500))


It is a truth universally acknowledged, was continued by the
no state of years ottld; she hated to look as her
consent to herself that Mrs. Bingness had already answered; but when a few minutiance of
the last new fumaling ease in general kinding a balls occurred hardly only that would have left
under her eyes and they call togethers.  Their unoblectioned the
visitial letters were suspecting it that she had not the Menward's ideas of accoutingly
name, and everything himself--but her admiration of that one on their
own hearts not


In [ ]:
model.save("janeausten_full.keras")